In [1]:
import pandas as pd
qbhalf = pd.read_pickle("../PickleFiles/QBs_HalfPPR.pkl")
rbhalf = pd.read_pickle("../PickleFiles/RBs_HalfPPR.pkl")
wrtehalf = pd.read_pickle("../PickleFiles/WRTE_HalfPPR.pkl")

qbnon = pd.read_pickle("../PickleFiles/QBs_NonPPR.pkl")
rbnon = pd.read_pickle("../PickleFiles/RBs_NonPPR.pkl")
wrtenon = pd.read_pickle("../PickleFiles/WRTE_NonPPR.pkl")

qbfull = pd.read_pickle("../PickleFiles/QBs_PPR.pkl")
rbfull = pd.read_pickle("../PickleFiles/RBs_PPR.pkl")
wrtefull = pd.read_pickle("../PickleFiles/WRTE_PPR.pkl")

halfrankings = pd.concat([qbhalf, rbhalf, wrtehalf], ignore_index=True)
nonrankings = pd.concat([qbnon, rbnon, wrtenon], ignore_index=True)
fullrankings = pd.concat([qbfull, rbfull, wrtefull], ignore_index=True)

halfrankings = halfrankings.sort_values(by='Final PPG', ascending=False)
nonrankings = nonrankings.sort_values(by='Final PPG', ascending=False)
fullrankings = fullrankings.sort_values(by='Final PPG', ascending=False)

halfrankings = halfrankings.reset_index(drop=True)
nonrankings = nonrankings.reset_index(drop=True)
fullrankings = fullrankings.reset_index(drop=True)

fullrankings

/Users/kmaran3/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Rank,Name,Team,Position,Final PPG
0,1,Bryce Young,CAR,QB,19.553426
1,2,C.J. Stroud,HOU,QB,19.411734
2,3,Skylar Thompson,MIA,QB,18.933045
3,4,Tommy DeVito,NYG,QB,18.892343
4,5,Aidan O'Connell,LV,QB,18.873477
...,...,...,...,...,...
307,77,D'Onta Foreman,CLE,RB,2.311901
308,49,Tyrod Taylor,NYJ,QB,2.061322
309,50,Joe Flacco,IND,QB,1.847602
310,51,Josh Johnson,BAL,QB,1.635752


In [2]:
corrections = {
    'Indianapolis Colts': 'IND',
    'New England Patriots': 'NE',
    'Tampa Bay Buccaneers': 'TB',
    'New Orleans Saints': 'NO',
    'Los Angeles Rams': 'LA',
    'St. Louis Rams': 'LA',
    'Arizona Cardinals': 'ARI',
    'Green Bay Packers': 'GB',
    'Houston Texans': 'HOU',
    'Las Vegas Raiders': 'LV',
    'Oakland Raiders': 'LV',
    'Baltimore Ravens': 'BAL',
    'Los Angeles Chargers': 'LAC',
    'San Diego Chargers': 'LAC',
    'Kansas City Chiefs': 'KC',
    'Tennessee Titans': 'TEN',
    'San Francisco 49ers': 'SF',
    'Atlanta Falcons': 'ATL',
    'Buffalo Bills': 'BUF',
    'Carolina Panthers': 'CAR',
    'Chicago Bears': 'CHI',
    'Cincinnati Bengals': 'CIN',
    'Cleveland Browns': 'CLE',
    'Dallas Cowboys': 'DAL',
    'Denver Broncos': 'DEN',
    'Detroit Lions': 'DET',
    'Jacksonville Jaguars': 'JAX',
    'Miami Dolphins': 'MIA',
    'Minnesota Vikings': 'MIN',
    'New York Giants': 'NYG',
    'New York Jets': 'NYJ',
    'Philadelphia Eagles': 'PHI',
    'Pittsburgh Steelers': 'PIT',
    'Seattle Seahawks': 'SEA',
    'Washington Commanders': 'WAS',
    'Washington Redskins': 'WAS',
    'Washington Football Team': 'WAS'
}

reverse_corrections = {v: k for k, v in corrections.items()}

# Map the team names in the DataFrame
fullrankings['Team'] = fullrankings['Team'].map(reverse_corrections)
halfrankings['Team'] = halfrankings['Team'].replace(reverse_corrections)
nonrankings['Team'] = nonrankings['Team'].replace(reverse_corrections)

bye_weeks = {
    'Detroit Lions': 5,
    'San Diego Chargers': 5,
    'Los Angeles Chargers': 5,
    'Philadelphia Eagles': 5,
    'Tennessee Titans': 5,
    'Kansas City Chiefs': 6,
    'St. Louis Rams': 6,
    'Los Angeles Rams': 6,
    'Miami Dolphins': 6,
    'Minnesota Vikings': 6,
    'Chicago Bears': 7,
    'Dallas Cowboys': 7,
    'Pittsburgh Steelers': 9,
    'San Francisco 49ers': 9,
    'Cleveland Browns': 10,
    'Green Bay Packers': 10,
    'Oakland Raiders': 10,
    'Las Vegas Raiders': 10,
    'Seattle Seahawks': 10,
    'Arizona Cardinals': 11,
    'Carolina Panthers': 11,
    'New York Giants': 11,
    'Tampa Bay Buccaneers': 11,
    'Atlanta Falcons': 12,
    'Buffalo Bills': 12,
    'Cincinnati Bengals': 12,
    'Jacksonville Jaguars': 12,
    'New Orleans Saints': 12,
    'New York Jets': 12,
    'Baltimore Ravens': 14,
    'Denver Broncos': 14,
    'Houston Texans': 14,
    'Indianapolis Colts': 14,
    'New England Patriots': 14,
    'Washington Football Team': 14,
    'Washington Commanders': 14
}

# Function to add bye weeks to DataFrame
def add_bye_weeks(df, bye_weeks):
    df['Bye Week'] = df['Team'].map(bye_weeks)
    return df

halfrankings = add_bye_weeks(halfrankings, bye_weeks)
nonrankings = add_bye_weeks(nonrankings, bye_weeks)
fullrankings = add_bye_weeks(fullrankings, bye_weeks)

fullrankings['Rank'] = range(1, len(fullrankings) + 1)
halfrankings['Rank'] = range(1, len(halfrankings) + 1)
nonrankings['Rank'] = range(1, len(nonrankings) + 1)

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the webpage
url = 'https://www.footballguys.com/adp'

# Fetch the HTML content from the URL
response = requests.get(url)
if response.status_code == 200:
    html_content = response.text

    # Create a BeautifulSoup object
    soup = BeautifulSoup(html_content, 'html.parser')

    # List to hold player names and their ESPN values
    player_data = []
    rows = soup.find_all('tr')

    for row in rows:
        # Extract the player name
        name_td = row.find('td', class_='name sticky-col text-start')
        if name_td:
            name_a = name_td.find('a')
            if name_a:
                player_name = name_a.get_text().strip()
                # Extract the 9th <td> element (index 8) for ESPN value
                tds = row.find_all('td')
                if len(tds) >= 9:
                    espn_value = tds[8].get_text().strip()
                    # Append the player name and ESPN value as a tuple to the list
                    player_data.append((player_name, espn_value))

    # Create a DataFrame from the list
    ESPN = pd.DataFrame(player_data, columns=['Player Name', 'ESPN ADP'])

    # Convert ESPN Rank to numeric for proper sorting, handling non-numeric cases
    ESPN['ESPN ADP'] = pd.to_numeric(ESPN['ESPN ADP'], errors='coerce')
    ESPN = ESPN.dropna().sort_values('ESPN ADP')  # Drop rows where conversion failed and sort

    # Print the DataFrame
    print(ESPN)
else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")


Empty DataFrame
Columns: [Player Name, ESPN ADP]
Index: []


In [4]:
fullrankings = fullrankings.merge(ESPN, left_on='Name', right_on='Player Name', how='left').drop(columns=['Player Name'])
halfrankings = halfrankings.merge(ESPN, left_on='Name', right_on='Player Name', how='left').drop(columns=['Player Name'])
nonrankings = nonrankings.merge(ESPN, left_on='Name', right_on='Player Name', how='left').drop(columns=['Player Name'])

In [5]:
fullrankings.to_pickle("../PickleFiles/Full PPR Rankings.pkl")
halfrankings.to_pickle("../PickleFiles/Half PPR Rankings.pkl")
nonrankings.to_pickle("../PickleFiles/Non PPR Rankings.pkl")